# Supplementary Figure S5 - WIOD Portability Example

Post hoc enrichment-based annotation of the WIOD 2014 country-sector input-output matrix. Rows are origin country-sectors, columns are destination country-sectors, and values are log1p-transformed intermediate flows. The notebook renders the annotated matrix, a condensed dendrogram, and a descriptive destination-flow summary for selected country-labeled clusters.

## Setup

Enable inline plotting and define the figure export and scratch paths.

In [ ]:
# Enable inline plotting for notebooks
%matplotlib inline

In [ ]:
import re
import zipfile
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadr
import textwrap

NB_ID = "supp_fig_5"
PNG_DIR = Path("png") / NB_ID
NB_SCRATCH_DIR = Path("scratch") / NB_ID
for d in (PNG_DIR, NB_SCRATCH_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"PNG output -> {PNG_DIR}")
print(f"Scratch output -> {NB_SCRATCH_DIR}")

## Load WIOD

Download or reuse the cached WIOD 2016 release archive and extract the 2014 input-output table.


In [ ]:
import os
import shutil

WIOD_DATAVERSE_FILE_ID = 199101  # WIOTS_in_R.zip, resolved via the Dataverse file-listing API
WIOD_ARCHIVE_URL = f"https://dataverse.nl/api/access/datafile/{WIOD_DATAVERSE_FILE_ID}"
WIOD_ARCHIVE_PATH = NB_SCRATCH_DIR / "WIOTS_in_R.zip"
WIOD_EXPECTED_SIZE_BYTES = 641_578_409  # confirmed via Dataverse API file listing (prior audit)
WIOD_YEAR = 2014
WIOD_RDATA_FILENAME = f"WIOT{WIOD_YEAR}_October16_ROW.RData"
WIOD_RDATA_PATH = NB_SCRATCH_DIR / WIOD_RDATA_FILENAME

# Repo-local places a copy of the archive might already sit, checked before falling
# back to a network download.
WIOD_ARCHIVE_CACHE_CANDIDATES = [
    WIOD_ARCHIVE_PATH,
    Path("data/interdisciplinary/WIOTS_in_R.zip"),
    Path("data/interdisciplinary/wiod/WIOTS_in_R.zip"),
    Path("data/wiod/WIOTS_in_R.zip"),
]
WIOD_ARCHIVE_ENV_PATH = os.environ.get("WIOD_ARCHIVE_PATH")  # optional, user-supplied override
if WIOD_ARCHIVE_ENV_PATH:
    WIOD_ARCHIVE_CACHE_CANDIDATES.append(Path(WIOD_ARCHIVE_ENV_PATH))


def _valid_cache_candidate(path):
    return path.is_file() and path.stat().st_size == WIOD_EXPECTED_SIZE_BYTES


cached_candidate = next(
    (p for p in WIOD_ARCHIVE_CACHE_CANDIDATES if _valid_cache_candidate(p)), None
)
if cached_candidate == WIOD_ARCHIVE_PATH:
    print(f"Using cached archive: {WIOD_ARCHIVE_PATH} ({WIOD_ARCHIVE_PATH.stat().st_size:,} bytes)")
elif cached_candidate is not None:
    print(f"Copied local archive cache: {cached_candidate} -> {WIOD_ARCHIVE_PATH}")
    shutil.copy2(cached_candidate, WIOD_ARCHIVE_PATH)
else:
    import urllib.request

    print(
        f"No valid local archive cache found; downloading {WIOD_ARCHIVE_URL} -> {WIOD_ARCHIVE_PATH}"
    )
    urllib.request.urlretrieve(WIOD_ARCHIVE_URL, WIOD_ARCHIVE_PATH)

print(f"Archive size:     {WIOD_ARCHIVE_PATH.stat().st_size:,} bytes")
print(f"Expected size:    {WIOD_EXPECTED_SIZE_BYTES:,} bytes")

# Extract only the single 2014 RData member -- not the full 15-year archive.
# A truncated or wrong-release archive would otherwise surface as an opaque KeyError.
with zipfile.ZipFile(WIOD_ARCHIVE_PATH) as zf:
    if WIOD_RDATA_FILENAME not in zf.namelist():
        raise FileNotFoundError(
            f"{WIOD_RDATA_FILENAME} not found in {WIOD_ARCHIVE_PATH}. Members: {zf.namelist()}"
        )
    if not WIOD_RDATA_PATH.exists():
        zf.extract(WIOD_RDATA_FILENAME, path=NB_SCRATCH_DIR)

print(f"Extracted member: {WIOD_RDATA_FILENAME}")
print(f"Extracted size:   {WIOD_RDATA_PATH.stat().st_size:,} bytes")

## Inspect Source Table

Read the RData object and confirm its row and column structure before building the matrix.


In [ ]:
rdata_result = pyreadr.read_r(str(WIOD_RDATA_PATH))
wiot = rdata_result["wiot"]

print(f"Raw loaded shape: {wiot.shape}")
print(f"Columns (first 8): {list(wiot.columns[:8])}")
print(f"Columns (last 5):  {list(wiot.columns[-5:])}")
print(f"Unique countries ({wiot['Country'].nunique()}): {sorted(wiot['Country'].unique())}")

## Build Intermediate-Flow Matrix

Extract the square country-sector by country-sector intermediate-use block. Rows are producers/sellers; columns are users/buyers. Final-demand columns, totals, value-added, and tax rows are excluded. Sectors T (Household activities) and U (Extraterritorial organizations) are also excluded: they are WIOD accounting placeholders rather than real production sectors -- U is exactly zero for all 44 countries on both axes, and T is exactly zero for 30 of 44 countries with negligible nonzero values elsewhere. After that exclusion, any remaining country-sector row or column with exactly zero total flow is also dropped: these are individual reporting gaps in WIOD (mostly professional/service sectors -- architecture, scientific research, advertising, insurance, publishing, postal -- for specific countries), not economic signal, and left in they form a second spurious all-zero block through the matrix.


In [ ]:
META_COLS = ["IndustryCode", "IndustryDescription", "Country", "RNr", "Year"]

# Sectors excluded as non-production accounting placeholders (see markdown above).
EXCLUDED_SECTORS = ("T", "U")

# Row selection: all non-TOT rows are intermediate-industry rows (RNr 1-56 by construction).
# Also drop the excluded placeholder sectors so they don't form a spurious all-zero
# cross through the matrix.
row_mask = (wiot["Country"] != "TOT") & (~wiot["IndustryCode"].isin(EXCLUDED_SECTORS))
row_meta = wiot.loc[row_mask, META_COLS].copy()
row_labels = (row_meta["Country"] + "_" + row_meta["IndustryCode"]).tolist()

# Column selection: per-country columns with numeric suffix 1-56 only (excludes the
# per-country final-demand columns 57-61 and the single TOT column), and excludes
# the same placeholder sectors from the column side.
data_cols = [c for c in wiot.columns if c not in META_COLS]
col_pattern = re.compile(r"^([A-Z]{3})(\d+)$")

# RNr -> IndustryCode map (positional: sector order is identical across countries),
# built from the full (unfiltered) metadata so it covers all 56 sectors regardless
# of the row exclusion above.
full_row_meta = wiot.loc[wiot["Country"] != "TOT", META_COLS]
rnr_to_industry_code = dict(
    zip(
        full_row_meta.loc[full_row_meta["Country"] == "AUS", "RNr"].astype(int),
        full_row_meta.loc[full_row_meta["Country"] == "AUS", "IndustryCode"],
    )
)
excluded_rnrs = {rnr for rnr, code in rnr_to_industry_code.items() if code in EXCLUDED_SECTORS}

intermediate_cols = []
for c in data_cols:
    m = col_pattern.match(c)
    if m and 1 <= int(m.group(2)) <= 56 and int(m.group(2)) not in excluded_rnrs:
        intermediate_cols.append(c)

print(f"Intermediate rows: {row_mask.sum()}")
print(f"Intermediate cols: {len(intermediate_cols)}")
print(f"Excluded sectors:  {EXCLUDED_SECTORS} (RNr {sorted(excluded_rnrs)})")

col_labels = []
for c in intermediate_cols:
    m = col_pattern.match(c)
    country, num = m.group(1), int(m.group(2))
    col_labels.append(f"{country}_{rnr_to_industry_code[num]}")

# Slice the square intermediate flow block. Row order follows the file's native order;
# columns are relabeled to match and reordered to align with the row order so the
# matrix is a proper square country-sector x country-sector block.
flow_block = wiot.loc[row_mask, intermediate_cols].copy()
flow_block.index = row_labels
flow_block.columns = col_labels
flow_block = flow_block[row_labels]  # align column order to row order

flow_block = flow_block.apply(pd.to_numeric, errors="coerce")

print(f"Intermediate flow block shape (pre zero-sum drop): {flow_block.shape}")

# Drop any remaining row or column with exactly zero total flow (individual WIOD
# reporting gaps, not sector-wide placeholders like T/U). Applied symmetrically to
# both axes so the block stays square.
zero_rows = set(flow_block.index[flow_block.sum(axis=1) == 0])
zero_cols = set(flow_block.columns[flow_block.sum(axis=0) == 0])
ZERO_SUM_EXCLUDED = sorted(zero_rows | zero_cols)
keep_labels = [label for label in row_labels if label not in set(ZERO_SUM_EXCLUDED)]

print(
    f"Zero-sum rows: {len(zero_rows)} | Zero-sum cols: {len(zero_cols)} | Union dropped: {len(ZERO_SUM_EXCLUDED)}"
)

flow_block = flow_block.loc[keep_labels, keep_labels]
row_labels = keep_labels
row_meta = (
    row_meta.set_index(row_meta["Country"] + "_" + row_meta["IndustryCode"])
    .loc[row_labels]
    .reset_index(drop=True)
)

print(f"Intermediate flow block shape: {flow_block.shape}")
print(f"Square: {flow_block.shape[0] == flow_block.shape[1]}")
print(
    f"Value range: [{flow_block.values.min():,.1f}, {flow_block.values.max():,.1f}] (million USD)"
)

## Build Annotation Mapping

Use native WIOD metadata only: country labels. Sector (ISIC Rev.4) labels are also derived here for destination-column display purposes but are not used as an enrichment annotation.

In [ ]:
sector_meta = (
    row_meta.assign(label=row_labels)
    .drop_duplicates(subset="label")
    .set_index("label")
    .loc[row_labels]
)

sector_to_labels = defaultdict(set)
country_to_labels = defaultdict(set)
for label, industry_code, country in zip(
    sector_meta.index, sector_meta["IndustryCode"], sector_meta["Country"]
):
    sector_to_labels[industry_code].add(label)
    country_to_labels[country].add(label)

print(f"ISIC sector terms: {len(sector_to_labels)}")
print(f"Country terms:     {len(country_to_labels)}")

## Transform Matrix

Apply a single log1p transform and build the HiMaLAYAS matrix object.


In [ ]:
log_flow_df = pd.DataFrame(
    np.log1p(flow_block.values),
    index=flow_block.index,
    columns=flow_block.columns,
)

import himalayas
from himalayas import Analysis, Annotations, Matrix

print(f"HiMaLAYAS version: {himalayas.__version__}")

matrix = Matrix(log_flow_df)
print(f"Matrix shape: {matrix.values.shape[0]:,} × {matrix.values.shape[1]:,}")

## Cluster and Enrich

Cluster row flow profiles with the standard HiMaLAYAS chain, then test country annotations with Benjamini-Hochberg FDR correction.

In [ ]:
annotation_dict = country_to_labels
annotations = Annotations(annotation_dict, matrix)
print(f"Annotation terms retained: {len(annotations.terms)}")

N_ROWS = matrix.values.shape[0]
MIN_CLUSTER_SIZE = 20  # conservative, simple value: ~0.8% of N_ROWS

print(f"Row count: {N_ROWS}")
print(f"min_cluster_size: {MIN_CLUSTER_SIZE}")

analysis = Analysis(matrix, annotations).cluster(
    linkage_method="ward",
    linkage_metric="euclidean",
    linkage_threshold="auto",
    optimal_ordering=True,
    min_cluster_size=MIN_CLUSTER_SIZE,
)

analysis = analysis.enrich(min_overlap=2).finalize(col_cluster=True)
results = analysis.results
results_sig = results.filter("qval <= 0.05")

print(f"All enriched rows: {len(results.df):,}")
print(f"Significant cluster-term pairs (q<=0.05): {len(results_sig.df):,}")
print(results_sig.df.head(10))

In [ ]:
COUNTRY_NAME = {
    "AUS": "Australia",
    "AUT": "Austria",
    "BEL": "Belgium",
    "BGR": "Bulgaria",
    "BRA": "Brazil",
    "CAN": "Canada",
    "CHE": "Switzerland",
    "CHN": "China",
    "CYP": "Cyprus",
    "CZE": "Czech Republic",
    "DEU": "Germany",
    "DNK": "Denmark",
    "ESP": "Spain",
    "EST": "Estonia",
    "FIN": "Finland",
    "FRA": "France",
    "GBR": "United Kingdom",
    "GRC": "Greece",
    "HRV": "Croatia",
    "HUN": "Hungary",
    "IDN": "Indonesia",
    "IND": "India",
    "IRL": "Ireland",
    "ITA": "Italy",
    "JPN": "Japan",
    "KOR": "South Korea",
    "LTU": "Lithuania",
    "LUX": "Luxembourg",
    "LVA": "Latvia",
    "MEX": "Mexico",
    "MLT": "Malta",
    "NLD": "Netherlands",
    "NOR": "Norway",
    "POL": "Poland",
    "PRT": "Portugal",
    "ROU": "Romania",
    "ROW": "Rest of world",
    "RUS": "Russia",
    "SVK": "Slovakia",
    "SVN": "Slovenia",
    "SWE": "Sweden",
    "TUR": "Turkey",
    "TWN": "Taiwan",
    "USA": "United States",
}

# Short display labels for the official WIOD/ISIC sector descriptions. Kept as the full
# 56-sector reference map, including the T/U placeholders excluded from the matrix.
SECTOR_ABBREV = {
    "A01": "Crop/animal production",
    "A02": "Forestry/logging",
    "A03": "Fishing/aquaculture",
    "B": "Mining/quarrying",
    "C10-C12": "Food/beverages/tobacco",
    "C13-C15": "Textiles/apparel/leather",
    "C16": "Wood products",
    "C17": "Paper products",
    "C18": "Printing",
    "C19": "Coke/refined petroleum",
    "C20": "Chemicals",
    "C21": "Pharmaceuticals",
    "C22": "Rubber/plastics",
    "C23": "Non-metallic minerals",
    "C24": "Basic metals",
    "C25": "Fabricated metal products",
    "C26": "Computer/electronic/optical",
    "C27": "Electrical equipment",
    "C28": "Machinery/equipment",
    "C29": "Motor vehicles",
    "C30": "Other transport equipment",
    "C31_C32": "Furniture/other manufacturing",
    "C33": "Repair/installation",
    "D35": "Electricity/gas/utilities",
    "E36": "Water supply",
    "E37-E39": "Waste management",
    "F": "Construction",
    "G45": "Motor vehicle trade/repair",
    "G46": "Wholesale trade",
    "G47": "Retail trade",
    "H49": "Land transport",
    "H50": "Water transport",
    "H51": "Air transport",
    "H52": "Warehousing/transport support",
    "H53": "Postal/courier",
    "I": "Accommodation/food service",
    "J58": "Publishing",
    "J59_J60": "Media production/broadcasting",
    "J61": "Telecommunications",
    "J62_J63": "IT/computer services",
    "K64": "Financial services",
    "K65": "Insurance/pension",
    "K66": "Financial auxiliary services",
    "L68": "Real estate",
    "M69_M70": "Legal/accounting/management",
    "M71": "Architectural/engineering",
    "M72": "Scientific research",
    "M73": "Advertising/market research",
    "M74_M75": "Other professional/scientific",
    "N": "Administrative/support services",
    "O84": "Public admin/defence",
    "P85": "Education",
    "Q": "Health/social work",
    "R_S": "Other services",
    "T": "Household activities",
    "U": "Extraterritorial organizations",
}


def readable_destination_label(code):
    """Convert a Country_SectorCode destination label to 'Country name | Sector label'."""
    country, sector = code.split("_", 1)
    return f"{COUNTRY_NAME[country]} | {SECTOR_ABBREV[sector]}"

In [ ]:
cluster_labels = results_sig.cluster_labels(rank_by="q", label_mode="top_term")
cluster_labels_readable = cluster_labels.copy()


def typed_annotation_label(term):
    if term not in COUNTRY_NAME:
        raise ValueError(f"Unexpected WIOD top annotation term: {term}")
    return COUNTRY_NAME[term]


cluster_labels_readable["label_readable"] = cluster_labels_readable["label"].map(
    typed_annotation_label
)
CLUSTER_LABEL_OVERRIDES = dict(
    zip(cluster_labels_readable["cluster"].astype(int), cluster_labels_readable["label_readable"])
)

## Render Annotated Matrix

Show the clustered flow matrix with enrichment labels and a capped enrichment track.


In [ ]:
from himalayas.plot import Plotter

LABEL_COLOR = "black"
BACKGROUND_COLOR = "white"
DENDRO_COLOR = "#888888"
FONT = "Helvetica"
MATRIX_CMAP = "Reds"
ENRICHMENT_CMAP = "YlOrBr"

vmax = float(np.quantile(log_flow_df.values, 0.99))
enrichment_norm_max = 40.0

plotter = (
    Plotter(results_sig)
    .set_background(color=BACKGROUND_COLOR)
    .plot_dendrogram(
        axes=[0.0552, 0.075, 0.075, 0.84],
        data_pad=0,
        color=DENDRO_COLOR,
        linewidth=0.75,
    )
    .set_figure(
        figsize=(12, 18),
        subplots_adjust={"left": 0.13, "right": 0.73, "bottom": 0.075, "top": 0.915},
    )
    .plot_matrix(
        cmap=MATRIX_CMAP,
        vmin=0,
        vmax=vmax,
        outer_color=LABEL_COLOR,
        outer_lw=0.8,
    )
    .plot_matrix_axis_labels(
        xlabel="Destination country-sector",
        ylabel="Origin country-sector",
        font=FONT,
        fontsize=18,
        color=LABEL_COLOR,
        xlabel_pad=6.0,
        ylabel_pad=0.012,
    )
    .set_label_panel(
        axes=[0.75, 0.075, 0.25, 0.84],
        gutter_color=BACKGROUND_COLOR,
        text_pad=0.006,
    )
    .plot_cluster_labels_compact(
        overrides=CLUSTER_LABEL_OVERRIDES,
        rank_by="q",
        label_mode="top_term",
        cluster_span="line",
        cluster_span_gap=4,
        cluster_span_color=LABEL_COLOR,
        cluster_span_lw=2,
        cluster_span_alpha=0.8,
        cluster_span_cap_width=0.0,
        cluster_span_left_pad=0.1,
        cluster_span_right_pad=0.0,
        cluster_marker=None,
        label_prefix=None,
        line_shape="straight",
        line_style="solid",
        line_start="none",
        line_end="none",
        line_color=LABEL_COLOR,
        line_lw=1,
        line_alpha=0.8,
        max_words=30,
        wrap_text=True,
        wrap_width=40,
        overflow="wrap",
        font=FONT,
        fontsize=20,
        color=LABEL_COLOR,
        skip_unlabeled=False,
        label_fields=("label", "n"),
        omit_words=(),
        boundary_color=LABEL_COLOR,
        boundary_lw=0.8,
        boundary_alpha=0.8,
    )
    .plot_cluster_bar(
        cmap=ENRICHMENT_CMAP,
        norm=plt.Normalize(0, enrichment_norm_max),
        name="sigbar",
        title="Enrichment",
        width=0.1,
        left_pad=0.05,
        right_pad=0.0,
    )
    .set_label_track_order(("sigbar",))
    .add_colorbar(
        name="matrix",
        cmap=MATRIX_CMAP,
        norm=plt.Normalize(0, vmax),
        label="Intermediate flow\n(log1p-transformed million USD)",
        ticks=[0, vmax / 2, vmax],
    )
    .add_colorbar(
        name="enrichment",
        cmap=ENRICHMENT_CMAP,
        norm=plt.Normalize(0, enrichment_norm_max),
        label=r"Enrichment ($-\log_{10}q$)",
        ticks=[0, enrichment_norm_max / 2, enrichment_norm_max],
    )
    .plot_colorbars(
        ncols=2,
        height=0.02,
        gap=0.04,
        hpad=0.04,
        vpad=0.0,
        fontsize=20,
        font=FONT,
        color=LABEL_COLOR,
        border_color=LABEL_COLOR,
        border_width=0.8,
        border_alpha=0.9,
        tick_decimals=1,
    )
)

FIGURE_PATH = PNG_DIR / "wiod_2014_annotated_matrix.png"
plotter.save(FIGURE_PATH, dpi=300, bbox_inches="tight")
plotter.show()

print(f"Saved {FIGURE_PATH}")

## Render Condensed Dendrogram

Summarize the same clustering/enrichment result in a compact cluster-level dendrogram. Display labels use readable country names; raw terms remain unchanged in the output table.

In [ ]:
from himalayas.plot import plot_dendrogram_condensed

N_SIGNIFICANT_CLUSTERS = int(results_sig.df["cluster"].nunique())
N_TOTAL_LEAF_CLUSTERS = len(results_sig.cluster_layout().cluster_spans)
print(f"Significant clusters to label: {N_SIGNIFICANT_CLUSTERS}")
print(f"Total leaf clusters drawn:     {N_TOTAL_LEAF_CLUSTERS}")

condensed_dendrogram = plot_dendrogram_condensed(
    results_sig,
    rank_by="q",
    label_mode="top_term",
    label_overrides=CLUSTER_LABEL_OVERRIDES,
    figsize=(4.8, 0.29 * N_TOTAL_LEAF_CLUSTERS),
    sigbar_cmap="YlOrBr",
    sigbar_min_logp=0.0,
    sigbar_max_logp=enrichment_norm_max,
    sigbar_width=0.055,
    sigbar_height=0.80,
    font=FONT,
    fontsize=12,
    max_words=30,
    wrap_text=True,
    wrap_width=40,
    overflow="ellipsis",
    label_fields=("label", "n", "q"),
    label_color=LABEL_COLOR,
    label_left_pad=0.025,
    dendrogram_color=DENDRO_COLOR,
    dendrogram_lw=0.8,
    background_color=BACKGROUND_COLOR,
)

CONDENSED_DENDROGRAM_PATH = PNG_DIR / "wiod_2014_condensed_dendrogram.png"
condensed_dendrogram.save(
    CONDENSED_DENDROGRAM_PATH,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.03,
)
condensed_dendrogram.show()

print(f"Saved {CONDENSED_DENDROGRAM_PATH}")

## Summarize Destination Flows

For selected country-labeled clusters, sum raw intermediate flow to concrete cross-border destination country-sectors. Same-country and Rest of world destinations are excluded so the panel shows named destinations rather than domestic or aggregate flows.


In [ ]:
COUNTRY_TERM_RE = re.compile(r"^[A-Z]{3}$")
N_TOP_CLUSTERS = 12  # matches the 4x3 destination-bar grid below
N_TOP_DESTINATIONS = 5
EXCLUDED_DESTINATION_COUNTRIES = (
    "ROW",
)  # aggregate "Rest of world", not a concrete country-sector


def select_top_country_clusters(cluster_labels_df, n=N_TOP_CLUSTERS):
    """Return the n most significant clusters whose top term is a country code."""
    is_country = cluster_labels_df["label"].str.fullmatch(COUNTRY_TERM_RE)
    country_clusters = cluster_labels_df[is_country].sort_values("qval")
    return country_clusters.head(n).reset_index(drop=True)


def summarize_top_destinations(flow_block, clusters, cluster_id, country, top_n=N_TOP_DESTINATIONS):
    """Return top raw-flow destinations, excluding domestic and ROW columns."""
    origin_rows = sorted(clusters.cluster_to_labels[cluster_id])
    dest_totals = flow_block.loc[origin_rows].sum(axis=0)
    excluded_prefixes = (f"{country}_",) + tuple(f"{c}_" for c in EXCLUDED_DESTINATION_COUNTRIES)
    concrete = dest_totals[~dest_totals.index.str.startswith(excluded_prefixes)]
    return concrete.sort_values(ascending=False).head(top_n)


top_country_clusters = select_top_country_clusters(cluster_labels)
print(f"Selected {len(top_country_clusters)} top country-labeled clusters:")
print(top_country_clusters[["cluster", "label", "qval"]].to_string(index=False))

In [ ]:
DEST_BAR_COLOR = "#333333"  # "#E8751A"
DEST_LABEL_COLOR = "black"
DEST_BACKGROUND_COLOR = "white"

DEST_FIGSIZE = (15, 22)
DEST_TITLE_FONTSIZE = 22
DEST_SUBTITLE_FONTSIZE = 18
DEST_SUBTITLE_GAP = 0.97
DEST_TITLE_BLOCK_PAD = 30
DEST_TITLE_BLOCK_LIFT = 0.05
DEST_BAR_HEIGHT = 0.75
DEST_LABEL_MAX_CHARS = 18
DEST_SECTOR_LINE_MAX_CHARS = 18
DEST_XLABEL_FONTSIZE = 18
DEST_XTICK_FONTSIZE = 16
DEST_YTICK_FONTSIZE = 18
DEST_SUBPLOT_WSPACE = 1.0
DEST_SUBPLOT_HSPACE = 0.3


def wrap_destination_label(
    label, max_chars=DEST_LABEL_MAX_CHARS, sector_line_max_chars=DEST_SECTOR_LINE_MAX_CHARS
):
    """Shorten and wrap a 'Country | Sector' label to fit a narrow bar-chart y-tick.

    Two-tier wrap, both tiers breaking only at existing word boundaries (never
    mid-word):
    1. Sector keeps only its first slash-delimited word (e.g.
       'Food/beverages/tobacco' -> 'Food'). If "Country | Sector" is still longer
       than max_chars, wrap once at the ' | ' separator so country and sector each
       land on their own line.
    2. If that sector line is itself still longer than sector_line_max_chars (e.g.
       'Financial auxiliary services'), wrap it again at the nearest space at or
       before the limit, continuing onto a third line if needed.
    """
    country, sector = label.split(" | ", 1)
    short_sector = sector.split("/")[0]
    combined = f"{country} | {short_sector}"
    if len(combined) <= max_chars:
        return combined

    if len(short_sector) <= sector_line_max_chars:
        return f"{country} |\n{short_sector}"

    sector_lines = textwrap.wrap(short_sector, width=sector_line_max_chars)
    return f"{country} |\n" + "\n".join(sector_lines)


fig, axes = plt.subplots(
    4,
    3,
    figsize=DEST_FIGSIZE,
    squeeze=False,
)
fig.patch.set_facecolor(DEST_BACKGROUND_COLOR)
fig.subplots_adjust(
    left=0.13,
    right=0.98,
    top=0.95,
    bottom=0.035,
    wspace=DEST_SUBPLOT_WSPACE,
    hspace=DEST_SUBPLOT_HSPACE,
)

N_GRID_ROWS = axes.shape[0]

for panel_index, (ax, row) in enumerate(
    zip(axes.flat, top_country_clusters.itertuples(index=False))
):
    is_bottom_row = panel_index // axes.shape[1] == N_GRID_ROWS - 1
    ax.set_facecolor(DEST_BACKGROUND_COLOR)
    top_dest = summarize_top_destinations(flow_block, results_sig.clusters, row.cluster, row.label)

    bar_labels = [
        wrap_destination_label(readable_destination_label(lbl)) for lbl in top_dest.index
    ][::-1]
    bar_values = top_dest.values[::-1]
    ax.barh(
        bar_labels,
        bar_values,
        color=DEST_BAR_COLOR,
        height=DEST_BAR_HEIGHT,
        edgecolor=DEST_LABEL_COLOR,
        lw=1,
    )
    ax.tick_params(axis="y", pad=4, length=6, width=2)
    ax.set_xlim(0, float(bar_values.max()) * 1.08)
    ax.set_title(
        COUNTRY_NAME[row.label],
        fontsize=DEST_TITLE_FONTSIZE,
        color=DEST_LABEL_COLOR,
        fontname=FONT,
        pad=DEST_TITLE_BLOCK_PAD + DEST_TITLE_BLOCK_LIFT,
    )
    ax.text(
        0.5,
        DEST_SUBTITLE_GAP + DEST_TITLE_BLOCK_LIFT,
        f"($q$={row.qval:.1e})",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=DEST_SUBTITLE_FONTSIZE,
        color=DEST_LABEL_COLOR,
        fontname=FONT,
    )
    if is_bottom_row:
        ax.set_xlabel(
            "Raw intermediate flow\n(million USD)",
            fontsize=DEST_XLABEL_FONTSIZE,
            color=DEST_LABEL_COLOR,
            fontname=FONT,
        )
    for spine in ax.spines.values():
        spine.set_color(DEST_LABEL_COLOR)
    ax.tick_params(axis="x", labelsize=DEST_XTICK_FONTSIZE, colors=DEST_LABEL_COLOR)
    ax.tick_params(axis="y", labelsize=DEST_YTICK_FONTSIZE, colors=DEST_LABEL_COLOR)
    for tick_label in [*ax.get_xticklabels(), *ax.get_yticklabels()]:
        tick_label.set_fontname(FONT)

DEST_FIGURE_PATH = PNG_DIR / "wiod_2014_top_country_cluster_destinations.png"
fig.savefig(DEST_FIGURE_PATH, dpi=300, bbox_inches="tight", facecolor=DEST_BACKGROUND_COLOR)
plt.show()

print(f"Saved {DEST_FIGURE_PATH}")

## Caption / Claim Boundary

**Supplementary Figure S5. WIOD portability example.** This notebook supports one auxiliary claim: HiMaLAYAS can annotate hierarchical structure in a non-biological, dense economic matrix using native metadata. The WIOD result is a portability example, not an economic-discovery claim and not a validation of the yeast GI-PCC analysis. No robustness, null, multi-year, or competing-dataset analysis is performed here.
